# 02 — Kill Switch in Action

The `KillSwitch` is one of Phionyx's runtime governance primitives. It is
**fail-closed**: any evaluation error or out-of-band signal triggers a
shutdown that the agent cannot prompt-engineer its way past.

This notebook walks through the four documented trigger paths plus the
NaN-guard fail-closed branch, and inspects the tamper-evident event log.

**Public API used:** `phionyx_core.governance.kill_switch.KillSwitch`,
`KillSwitchConfig`, `KillSwitchTrigger`, `KillSwitchState`.


## 1. Imports

In [1]:
import math
from phionyx_core.governance.kill_switch import (
    KillSwitch,
    KillSwitchConfig,
    KillSwitchTrigger,
    KillSwitchState,
)

print("KillSwitch loaded.")
print("Triggers:", [t.value for t in KillSwitchTrigger])
print("States:  ", [s.value for s in KillSwitchState])


KillSwitch loaded.
Triggers: ['ethics_max_risk_exceeded', 't_meta_below_threshold', 'consecutive_drift_exceeded', 'manual_trigger', 'evaluation_error']
States:   ['armed', 'triggered', 'cooldown', 'disarmed']


## 2. Default config and a normal turn

With default thresholds (ethics_max_risk=0.95, t_meta_min=0.1,
consecutive_drift_max=5), a healthy turn does not trigger.


In [2]:
ks = KillSwitch()
print("initial state:", ks.state.value)
print("is_armed:    ", ks.is_armed)
print("is_triggered:", ks.is_triggered)

# Healthy turn: low ethics risk, high meta-trust, no drift
result = ks.evaluate(ethics_max_risk=0.10, t_meta=0.85, drift_detected=False, turn_id=1)
print(f"\nturn 1 -> triggered={result.triggered}  reason={result.reason!r}")


initial state: armed
is_armed:     True
is_triggered: False

turn 1 -> triggered=False  reason='All safety checks passed'


## 3. Trigger 1 — Catastrophic ethics violation

The `ethics_max_risk` argument is a 0–1 score from the deliberative
ethics block. Anything above 0.95 trips the gate immediately.


In [3]:
ks = KillSwitch()
result = ks.evaluate(ethics_max_risk=0.97, t_meta=0.8, drift_detected=False, turn_id=10)

print("triggered:", result.triggered)
print("trigger:  ", result.trigger.value if result.trigger else None)
print("reason:   ", result.reason)
print("\nks.state:", ks.state.value)
print("subsequent calls return the locked state:")
print(ks.evaluate(ethics_max_risk=0.0, t_meta=1.0, turn_id=11).reason)


KILL SWITCH TRIGGERED: ethics_max_risk_exceeded — Ethics max risk 0.970 exceeds threshold 0.95


triggered: True
trigger:   ethics_max_risk_exceeded
reason:    Ethics max risk 0.970 exceeds threshold 0.95

ks.state: triggered
subsequent calls return the locked state:
Kill switch already triggered — awaiting manual reset


## 4. Trigger 2 — Meta-cognitive trust collapse (T_meta < 0.1)

`t_meta` is the runtime's own confidence in its judgments. If it falls
below 0.1 we treat the agent as no longer trustworthy and shut down.


In [4]:
ks = KillSwitch()
result = ks.evaluate(ethics_max_risk=0.0, t_meta=0.05, drift_detected=False, turn_id=20)
print("triggered:", result.triggered, " trigger:", result.trigger.value)
print("reason:   ", result.reason)


KILL SWITCH TRIGGERED: t_meta_below_threshold — T_meta 0.050 below minimum threshold 0.1


triggered: True  trigger: t_meta_below_threshold
reason:    T_meta 0.050 below minimum threshold 0.1


## 5. Trigger 3 — Sustained behavioral drift

A single drift signal does not trip the gate. The configured threshold
(default 5 consecutive turns) does. This prevents single-turn noise from
faulting the system, but stops a slow-drifting agent.


In [5]:
ks = KillSwitch(KillSwitchConfig(consecutive_drift_max=5))

for turn in range(1, 8):
    r = ks.evaluate(ethics_max_risk=0.0, t_meta=0.9, drift_detected=True, turn_id=turn)
    state = ks.state.value
    print(f"turn {turn:>2}  drift_count={ks._consecutive_drift_count:>2}  "
          f"state={state:<10}  triggered={r.triggered}")
    if r.triggered:
        print(f"  -> trigger={r.trigger.value} reason={r.reason!r}")
        break


KILL SWITCH TRIGGERED: consecutive_drift_exceeded — Consecutive drift count 6 exceeds max 5


turn  1  drift_count= 1  state=armed       triggered=False
turn  2  drift_count= 2  state=armed       triggered=False
turn  3  drift_count= 3  state=armed       triggered=False
turn  4  drift_count= 4  state=armed       triggered=False
turn  5  drift_count= 5  state=armed       triggered=False
turn  6  drift_count= 6  state=triggered   triggered=True
  -> trigger=consecutive_drift_exceeded reason='Consecutive drift count 6 exceeds max 5'


## 6. Trigger 4 — NaN fail-closed guard

Numeric inputs are validated. A NaN slipping in (corrupt sensor, broken
upstream metric) triggers shutdown rather than being silently accepted.


In [6]:
ks = KillSwitch()
result = ks.evaluate(ethics_max_risk=float("nan"), t_meta=0.9, turn_id=30)
print("triggered:", result.triggered)
print("trigger:  ", result.trigger.value)
print("reason:   ", result.reason)


KILL SWITCH: NaN detected in ethics_max_risk — fail-closed


KILL SWITCH TRIGGERED: evaluation_error — NaN detected in ethics_max_risk (fail-closed)


triggered: True
trigger:   evaluation_error
reason:    NaN detected in ethics_max_risk (fail-closed)


## 7. Tamper-evident event log

Every evaluation appends to `_event_log`. In production the gate emits an
`AuditRecord` with an Ed25519 signature and a hash chain to the previous
record (see `phionyx_core.contracts.v4.audit_record`). Here we just inspect
the in-memory log.


In [7]:
ks = KillSwitch()

ks.evaluate(ethics_max_risk=0.10, t_meta=0.8, turn_id=1)
ks.evaluate(ethics_max_risk=0.20, t_meta=0.7, drift_detected=True, turn_id=2)
ks.evaluate(ethics_max_risk=0.97, t_meta=0.7, turn_id=3)  # trigger

print(f"events recorded: {len(ks.event_log)}\n")
for ev in ks.event_log:
    metrics_short = {k: round(v, 3) for k, v in ev.metrics.items()}
    after = ev.state_after.value
    trig = ev.trigger.value if ev.trigger else "-"
    print(f"  turn={ev.turn_id} state_after={after:<10} trigger={trig:<28} metrics={metrics_short}")


KILL SWITCH TRIGGERED: ethics_max_risk_exceeded — Ethics max risk 0.970 exceeds threshold 0.95


events recorded: 3

  turn=1 state_after=armed      trigger=-                            metrics={'ethics_max_risk': 0.1, 't_meta': 0.8, 'drift_detected': 0.0, 'consecutive_drift_count': 0.0}
  turn=2 state_after=armed      trigger=-                            metrics={'ethics_max_risk': 0.2, 't_meta': 0.7, 'drift_detected': 1.0, 'consecutive_drift_count': 0.0}
  turn=3 state_after=triggered  trigger=ethics_max_risk_exceeded     metrics={'ethics_max_risk': 0.97, 't_meta': 0.7, 'drift_detected': 0.0, 'consecutive_drift_count': 1.0}


## What this proves

- Triggers are bounded scalars, not prompt instructions.
- The gate is fail-closed by design — corrupt input triggers shutdown.
- Once tripped, every subsequent call returns the locked state. There is
  no "talk past it" path.
- Every evaluation is logged for offline review.

A 30-line safety primitive that does not depend on the model behaving
well is the whole point.
